# All-rundle pairwise agreement

Interactive front-end for `ll_all_rundle_analysis.py`. The expensive step (`build_agreement_tables`, via `get_tables`) runs once per season in this kernel session (cached); every cell after that is a cheap lookup against the precomputed tables, so you can re-run queries against different rundles/players without rebuilding anything.

Set `SEASON` below to `108` or `109` to switch which season everything in this notebook operates on -- the two are analyzed independently (no cross-season comparison).

In [1]:
import time

from ll_all_rundle_analysis import (
    get_tables,
    load_season_data,
    pairwise_agreement,
    one_pairwise_agreement,
    average_agreement,
    top_agreement,
    most_similar_pairs,
    least_similar_pairs,
    most_similar_player,
    agreement_dict_for_rundle,
)
from ll_analysis import print_agreement_matrix

In [2]:
SEASON = 109  # switch to 109 for LL109

data = load_season_data(SEASON)
print(f"season {data['season']}: {data['num_days']} days x {data['num_questions']} questions/day")
print(f"{len(data['rundles'])} rundle entries ({sum(1 for r in data['rundles'].values() if r)} non-empty)")

season 109: 25 days x 6 questions/day
1307 rundle entries (1307 non-empty)


In [3]:
t0 = time.time()
tables108 = get_tables(season=108)
print(f"built tables for {len(tables108)} (rundle, player) rows in {time.time() - t0:.1f}s")

built tables for 35858 (rundle, player) rows in 94.6s


In [10]:
t0 = time.time()
tables109 = get_tables(season=109)
print(f"built tables for {len(tables109)} (rundle, player) rows in {time.time() - t0:.1f}s")

built tables for 36224 (rundle, player) rows in 91.6s


## Pick a rundle to explore

Rundle names look like `A_Aloha`, `B_Beach`, etc. -- the letter prefix before the underscore is the tier/branch (`A` through `E`, plus `R` for Rookie).

In [ ]:
sorted(tables.index_by_rundle.keys())[:20]

In [ ]:
RUNDLE = sorted(tables.index_by_rundle.keys())[0]
print(f"using rundle: {RUNDLE}")
print_agreement_matrix(agreement_dict_for_rundle(tables, RUNDLE))

In [ ]:
print(f"Most similar pairs in {RUNDLE}:")
for (a, b), val in most_similar_pairs(tables, n=10, rundle=RUNDLE):
    print(f"  {a[1]} & {b[1]}: {val:.1%}")

print(f"\nLeast similar pairs in {RUNDLE}:")
for (a, b), val in least_similar_pairs(tables, n=10, rundle=RUNDLE):
    print(f"  {a[1]} & {b[1]}: {val:.1%}")

In [ ]:
print(f"Average agreement within {RUNDLE}:")
for label, avg in average_agreement(tables, rundle=RUNDLE):
    print(f"  {label[1]:<14} {avg:.1%}" if avg is not None else f"  {label[1]} n/a")

## Single player / single pair lookups

Names are usually unique, so plain `"LastFirst"` works; pass `rundle=` to disambiguate if a name happens to collide across rundles.

In [ ]:
PLAYER = tables.labels[tables.index_by_rundle[RUNDLE][0]][1]
print(f"Agreement between {PLAYER} and everyone in {RUNDLE}:")
for label, val in one_pairwise_agreement(tables, PLAYER, scope_rundle=RUNDLE):
    print(f"  {label[1]:<14} {val:.1%}")

## Whole-season queries

These scan across all ~36k rows instead of one rundle, so they're a lot heavier (multi-GB temporary arrays, tens of seconds). Run only if you have RAM headroom to spare.

In [3]:
t0 = time.time()
season_avgs = average_agreement(tables108)
print(f"computed in {time.time() - t0:.1f}s\n")
print("Top 10 average agreement (whole season):")
for label, avg in season_avgs[:10]:
    print(f"  {label[1]:<14} ({label[0]:<12}) {avg:.1%}" if avg is not None else f"  {label} n/a")

computed in 6.8s

Top 10 average agreement (whole season):
  WeissE2        (C_Oasis     ) 64.1%
  McLeanL        (C_Keystone  ) 64.0%
  VincentEC      (C_Memorial_Div_2) 63.9%
  HaberS2        (C_Junction_Div_1) 63.8%
  BlairJ         (B_Rainbow   ) 63.8%
  RedwineK       (D_River_Div_1) 63.7%
  CurranP2       (B_Sakura    ) 63.7%
  MeserveyM      (B_Tidewater ) 63.7%
  ChoateK        (D_Wilderness_Div_1) 63.7%
  GuytonA        (A_Horizon   ) 63.6%


In [4]:
t0 = time.time()
season_avgs = top_agreement(tables108)
print(f"computed in {time.time() - t0:.1f}s\n")
print("Top 10 top agreement (whole season):")
for label, avg in season_avgs[:10]:
    print(f"  {label[1]:<14} ({label[0]:<12}) {avg:.1%}" if avg is not None else f"  {label} n/a")
print("\nBottom 10 top agreement (whole season):")
for label, avg in season_avgs[-10:]:
    print(f"  {label[1]:<14} ({label[0]:<12}) {avg:.1%}" if avg is not None else f"  {label} n/a")

computed in 5.1s

Top 10 top agreement (whole season):
  CalhounC       (A_Patagonia ) 98.7%
  CalhounJ3      (A_Patagonia ) 98.7%
  SelzerE        (A_Forest    ) 96.0%
  KightR         (A_Polaris   ) 96.0%
  BlairT         (A_Cosmos    ) 95.3%
  HellendagI     (A_Tranquility) 95.3%
  OttolinoC      (A_Avalon    ) 94.7%
  ButschekAHeyHey (A_Cherry    ) 94.7%
  BarkerC        (A_Frontier  ) 94.7%
  MunkD          (A_Galaxy    ) 94.7%

Bottom 10 top agreement (whole season):
  SmithS28       (R_Div_71    ) 68.7%
  ArgentoM       (R_Div_76    ) 68.7%
  HerskowitzA    (D_Mojave_Div_1) 68.0%
  KuhtzM13       (E_Pampas_Div_2) 68.0%
  CreswellL      (B_Cypress   ) 67.3%
  MillerB10      (C_Sequoia_Div_2) 67.3%
  DallandC       (D_Boardwalk_Div_1) 67.3%
  RuntDR         (D_Vista_Div_2) 67.3%
  JonesJE        (E_Canyon_Div_1) 67.3%
  De GrootI      (E_Cherry_Div_1) 66.7%


In [6]:
top = top_agreement(tables108)
vals = [v for _, v in top if v is not None]
avg_top_agreement = sum(vals) / len(vals)
min_top_agreement = min(vals)  # == vals[-1], since sorted descending
median_top_agreement = vals[len(vals) // 2]
print(f"\nAverage top agreement (whole season): {avg_top_agreement:.1%}")
print(f"Median top agreement (whole season): {median_top_agreement:.1%}")


Average top agreement (whole season): 78.5%
Median top agreement (whole season): 78.0%


In [11]:
t0 = time.time()
top_pairs = most_similar_pairs(tables, n=10, min_non_forfeit=126)
print(f"computed in {time.time() - t0:.1f}s\n")
print("Top 10 most similar pairs (whole season):")
for (a, b), val in top_pairs:
    print(f"  {a[1]} ({a[0]}) & {b[1]} ({b[0]}): {val:.1%}")

computed in 73.0s

Top 10 most similar pairs (whole season):
  CalhounC (A_Patagonia) & CalhounJ3 (A_Patagonia): 99.3%
  PearlmanM (A_Patagonia) & RogersTL (A_Patagonia): 99.2%
  BerryC (A_Island) & IsmailST (A_Mojave): 96.0%
  VenguswamyK (A_Midland) & IsmailST (A_Mojave): 95.8%
  FrielP (A_Boardwalk) & FanoeG (A_Midland): 95.3%
  JacksonC8 (A_Kookaburra) & CrowleyA2 (A_Serengeti): 95.3%
  MunkD (A_Galaxy) & HessJa (A_Kaleidoscope): 95.3%
  Sanchez F (D_Memorial_Div_1) & ThomasE6 (D_Memorial_Div_1): 95.1%
  BlackMR (B_Wilderness) & BakerM34 (C_Wilderness_Div_1): 94.9%
  HessJa (A_Kaleidoscope) & FanoeG (A_Midland): 94.7%


In [7]:
one_pairwise_agreement(tables108, "JacksonM", min_non_forfeit=126)

[(('A_Nebula', 'LeeDK'), 0.8733333333333333),
 (('A_Kaleidoscope', 'HessJa'), 0.8733333333333333),
 (('A_Prairie', 'StevensN2'), 0.8666666666666667),
 (('A_Olympic', 'DhuwaliaR'), 0.8666666666666667),
 (('A_Olympic', 'KleinJT'), 0.8611111111111112),
 (('A_Metro', 'WebsterW'), 0.86),
 (('A_Rubicon', 'PlotkinD'), 0.86),
 (('A_Summit', 'LimAB'), 0.86),
 (('A_Ranger', 'SaferM'), 0.86),
 (('A_Maritime', 'AggarwalA'), 0.8560606060606061),
 (('A_Forest', 'ColwellB626'), 0.855072463768116),
 (('A_Badlands', 'ScheelerD'), 0.8541666666666666),
 (('A_Cosmos', 'BlairT'), 0.8533333333333334),
 (('A_Typhoon', 'RautY'), 0.8533333333333334),
 (('A_Galaxy', 'MunkD'), 0.8533333333333334),
 (('A_Oasis', 'ChiltonC2'), 0.8533333333333334),
 (('A_Rainforest', 'TorioJ'), 0.8533333333333334),
 (('A_Cardinal', 'ChakravarthyP'), 0.8533333333333334),
 (('A_Mojave', 'LloydP3'), 0.8484848484848485),
 (('A_Metro', 'WebsterB2'), 0.8472222222222222),
 (('A_Boardwalk', 'FrielP'), 0.8466666666666667),
 (('A_Wilderness'

In [13]:
most_similar_player(tables109, "MaD", min_non_forfeit=126)

(('B_Harbor', 'Dey ChowdhuryM'), 0.8066666666666666)

In [4]:
one_pairwise_agreement(tables109, "OzarowC", min_non_forfeit=126)

[(('A_Nebula', 'SimsC'), 0.8466666666666667),
 (('A_Nebula', 'SharpeR1'), 0.8333333333333334),
 (('B_Maritime', 'AggarwalA'), 0.8263888888888888),
 (('B_Evergreen', 'PyleA'), 0.82),
 (('A_Nebula', 'SchleckE'), 0.8133333333333334),
 (('B_Glacier', 'SunD'), 0.8133333333333334),
 (('B_Coastal', 'CatlinPG'), 0.8133333333333334),
 (('R_Div_48', 'IsmailA'), 0.8133333333333334),
 (('A_Centennial', 'Drnovsek ZorkoF'), 0.8066666666666666),
 (('B_Badlands', 'BaramI'), 0.8043478260869565),
 (('B_Citadel', 'ShuklaA'), 0.8015873015873016),
 (('A_Rainforest', 'Talvola V'), 0.8),
 (('C_Jungle_Div_2', 'LiflandA3'), 0.8),
 (('A_Peninsula', 'LinJK'), 0.8),
 (('B_Continental', 'ChandrashekarQ'), 0.7971014492753623),
 (('A_Central', 'WilliamsRM'), 0.7933333333333333),
 (('A_Sahara', 'NadigA'), 0.7933333333333333),
 (('B_Nautilus', 'BurchJ'), 0.7933333333333333),
 (('A_Cardinal', 'ArribasD'), 0.7933333333333333),
 (('A_Seaboard', 'LinA3'), 0.7933333333333333),
 (('C_Riviera', 'Hawkins-PottierG'), 0.7933333

In [13]:
one_pairwise_agreement(tables, "MaD", min_non_forfeit=126)

[(('B_Harbor', 'Dey ChowdhuryM'), 0.8066666666666666),
 (('C_Aloha_Div_1', 'KimPK'), 0.8),
 (('A_Byzantium', 'TanMac'), 0.8),
 (('B_Badlands', 'BaramI'), 0.7971014492753623),
 (('B_Nebula', 'OzarowC'), 0.7933333333333333),
 (('C_Nebula_Div_1', 'MandelbaumE2'), 0.7933333333333333),
 (('B_Evergreen', 'PyleA'), 0.7866666666666666),
 (('A_Nebula', 'SharpeR1'), 0.7866666666666666),
 (('A_Rainforest', 'Talvola V'), 0.78),
 (('A_Cosmos', 'ChernicoffS'), 0.78),
 (('B_Lighthouse', 'HuangJ404'), 0.7777777777777778),
 (('A_Orchid', 'LandolfiR'), 0.7777777777777778),
 (('B_Nautilus', 'BurchJ'), 0.7733333333333333),
 (('C_Byzantium', 'NepplH'), 0.7733333333333333),
 (('A_Village', 'LuE'), 0.7733333333333333),
 (('D_Olympic_Div_1', 'GuoAvocado'), 0.7727272727272727),
 (('C_Grove_Div_1', 'PradeepkumarV'), 0.7708333333333334),
 (('B_Orbital', 'EhresmannP'), 0.7698412698412699),
 (('C_Olive_Div_2', 'FreyerJ'), 0.7681159420289855),
 (('A_Sequoia', 'RajivS'), 0.7681159420289855),
 (('B_Coastal', 'CatlinP

In [28]:
pairwise_agreement(tables109, "OzarowC", "TallmanM")

0.5972222222222222

In [5]:
average_agreement(tables)[:10]

/Users/ceruleanozarow/Downloads/LL_Analysis_V2/ll_all_rundle_analysis.py:376: RuntimeWarning: Mean of empty slice
  row_avg = np.nanmean(frac, axis=1)


[(('E_Delta_Div_1', 'WeichertM'), 0.6647493839263916),
 (('D_Rainforest_Div_2', 'WilderC2'), 0.657676637172699),
 (('E_Frontier_Div_2', 'GarabaduR'), 0.657676637172699),
 (('C_Memorial_Div_2', 'VincentEC'), 0.6386399865150452),
 (('E_Pampas_Div_2', 'HuberC2'), 0.6384621262550354),
 (('C_Oasis', 'WeissE2'), 0.637787401676178),
 (('D_Outback_Div_2', 'KoenigS2'), 0.6371365785598755),
 (('E_Sunrise_Div_2', 'RandallA212'), 0.6359637379646301),
 (('C_Junction_Div_1', 'HaberS2'), 0.6355755925178528),
 (('B_Arctic', 'AndersonE31'), 0.6348219513893127)]

In [4]:
h = average_agreement(tables, min_non_forfeit=126)
h[:10]
h[-10:]

[(('E_Saguaro_Div_1', 'McCormickG'), 0.499182790517807),
 (('R_Div_45', 'MordarskiM'), 0.4987621009349823),
 (('A_Olive', 'JosephJ975'), 0.4985541105270386),
 (('E_Skyline_Div_1', 'AguilarJ2'), 0.49803969264030457),
 (('B_Cardinal', 'ReccaJ'), 0.4955570697784424),
 (('C_Foundry', 'ToemanG'), 0.49480578303337097),
 (('B_Lighthouse', 'EilbacherP'), 0.49401330947875977),
 (('E_Plaza_Div_1', 'GaisserNR'), 0.4916108548641205),
 (('E_Cherry_Div_1', 'De GrootI'), 0.4879300892353058),
 (('E_Cypress_Div_2', 'AttalJ'), 0.48614194989204407)]

In [5]:
h[:10]

[(('C_Memorial_Div_2', 'VincentEC'), 0.639046847820282),
 (('E_Pampas_Div_2', 'HuberC2'), 0.6388934850692749),
 (('C_Oasis', 'WeissE2'), 0.6387237310409546),
 (('D_Outback_Div_2', 'KoenigS2'), 0.6380500793457031),
 (('C_Junction_Div_1', 'HaberS2'), 0.6363679766654968),
 (('B_Arctic', 'AndersonE31'), 0.6362183690071106),
 (('R_Div_27', 'Turbyfill J'), 0.6345738172531128),
 (('D_Labyrinth_Div_2', 'EvansN'), 0.63394695520401),
 (('B_Rainbow', 'BlairJ'), 0.6336891055107117),
 (('C_Keystone', 'McLeanL'), 0.6336002945899963)]